# XGBoost — How It Actually Works
---

## Section 1: The big picture

XGBoost builds a strong model by adding **many small decision trees** one by one. Each new tree is a **correction** to the current predictions:

$$\hat{y}_i = \sum_{b=1}^{B} f_b(x_i)$$

Think of it as: **baseline prediction + correction₁ + correction₂ + ...**

This is fundamentally different from random forests, which build trees *independently* and average them. XGBoost builds trees *sequentially* — each one learns from the previous ensemble's mistakes.

Random forest trees are **generalists** — they all try to predict the target. XGBoost trees are **specialists** — each one focuses on the specific errors remaining. This usually gives better predictions, but it's riskier: because each tree is chasing errors, it can start chasing **noise** if you're not careful. That's why XGBoost needs guardrails that random forests don't.

XGBoost's three key advantages over "plain" gradient boosting:
1. **Explicit regularization** — penalizes complex trees to reduce overfitting
2. **Better split scoring** — uses both gradient + curvature (not just gradient)
3. **Strong engineering** — fast, handles missing/sparse values natively


## Section 2: What each new tree is trying to learn

At boosting step $b$, you already have predictions $\hat{y}_i^{(b-1)}$. For each data point, you ask: **"should my prediction go up or down, and by how much?"**

Two numbers answer this:

**$g_i = \frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i}$** (gradient) — the direction to change $\hat{y}$ to reduce loss.
- If $g_i > 0$: prediction is too high → push it down
- If $g_i < 0$: prediction is too low → push it up

**$h_i = \frac{\partial^2 L(y_i, \hat{y}_i)}{\partial \hat{y}_i^2}$** (hessian) — how sensitive the loss is around $\hat{y}$. Bigger $h$ = loss changes faster = be more careful with the correction.

### Quick example (squared error)

If $L(y, \hat{y}) = \frac{1}{2}(y - \hat{y})^2$, then $g_i = \hat{y}_i - y_i$ and $h_i = 1$.

If $y = 80$ and $\hat{y} = 100$, then $g = +20$ (positive) → the next tree should push the prediction **down**.

### Where do g and h come from?

They come from differentiating the **loss function only** — not the penalty. The regularization shows up later, when computing leaf values and deciding splits. This matters: the gradients tell you "where am I wrong" without any regularization influence.


## Section 3: The objective function — what it is and what it isn't

Every textbook introduces XGBoost with the objective:

$$\mathcal{L} = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \sum_{b=1}^{B} \Omega(f_b)$$

Where the regularizer for a tree is:

$$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2 \quad (\text{and sometimes } \alpha \sum |w_j|)$$

- $T$ = number of leaves (more leaves = more complex)
- $w_j$ = value predicted by leaf $j$
- $\gamma, \lambda, \alpha$ = regularization parameters that penalize complexity

### The honest framing

Your brain sees this equation and thinks "I compute this number, then optimize it" — the same way you'd compute a loss in deep learning and backprop through it.

**That's not what happens.** You never calculate the objective as one number. The equation is the *theoretical justification* for why the procedure works. When you take derivatives and solve for the optimal tree, the equation **dissolves** into specific rules:

- $g$ and $h$ come from differentiating the **Loss** part
- $\gamma$ and $\lambda$ come from differentiating the **Penalty** part
- They meet inside the tree-building decisions

The procedure is what runs. The equation explains why it works. Let's first walk through the full mathematical derivation, then the procedure.


## Section 3b: Full mathematical derivation

This section derives every formula XGBoost uses from the objective function. Each step includes the intuition for *why* we're doing it.

### Step 1: The objective at round $t$

At round $t$, predictions from the previous $t-1$ trees are fixed. We're choosing one new tree $f_t$:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

*Why this form:* $\hat{y}_i^{(t-1)}$ is locked — previous trees can't be changed. The only thing we're choosing is $f_t$. So the objective is a function of one variable: the new tree.

### Step 2: Taylor expansion (the key approximation)

Computing $L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i))$ exactly for every possible tree is intractable. So we approximate the loss using a second-order Taylor expansion around $\hat{y}_i^{(t-1)}$:

$$L\left(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\right) \approx L\left(y_i,\; \hat{y}_i^{(t-1)}\right) + g_i \cdot f_t(x_i) + \frac{1}{2} h_i \cdot f_t(x_i)^2$$

Where:

$$g_i = \frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i}\bigg|_{\hat{y}_i = \hat{y}_i^{(t-1)}} \qquad h_i = \frac{\partial^2 L(y_i, \hat{y}_i)}{\partial \hat{y}_i^2}\bigg|_{\hat{y}_i = \hat{y}_i^{(t-1)}}$$

*Why Taylor expansion:* it replaces the complex loss function with a **simple quadratic** in $f_t(x_i)$. A quadratic has a closed-form minimum — you can solve it with algebra instead of iterative optimization.

*For squared error:* $L = \frac{1}{2}(y_i - \hat{y}_i)^2$ gives $g_i = \hat{y}_i - y_i$ and $h_i = 1$. But the derivation works for **any** differentiable loss.

### Step 3: Simplify the objective

Since $L(y_i, \hat{y}_i^{(t-1)})$ is a constant (previous trees are frozen), we drop it:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i \cdot f_t(x_i) + \frac{1}{2} h_i \cdot f_t(x_i)^2 \right] + \Omega(f_t)$$

*What this says:* to find the best new tree, we don't need the full loss — just the gradient and hessian at each data point, plus the penalty on the tree.

### Step 4: Define the tree mathematically

A tree with $T$ leaves partitions the input space into $T$ regions $R_1, R_2, ..., R_T$. Each leaf $j$ outputs a constant value $w_j$:

$$f_t(x_i) = w_{q(x_i)}$$

Where $q: \mathbb{R}^d \rightarrow \{1, 2, ..., T\}$ maps data point $x_i$ to its leaf index.

### Step 5: Define the regularization

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

- $\gamma T$: penalty per leaf — more leaves = higher cost
- $\frac{1}{2}\lambda \sum w_j^2$: penalty on leaf values — large predictions = higher cost

### Step 6: Rewrite the objective by leaf (the key restructuring)

Instead of summing over data points, we sum over leaves. Define $I_j = \{i \mid q(x_i) = j\}$ as the set of data points in leaf $j$.

Since $f_t(x_i) = w_j$ for all $i \in I_j$:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ \left(\sum_{i \in I_j} g_i\right) w_j + \frac{1}{2}\left(\sum_{i \in I_j} h_i + \lambda\right) w_j^2 \right] + \gamma T$$

Define shorthand:

$$G_j = \sum_{i \in I_j} g_i \qquad H_j = \sum_{i \in I_j} h_i$$

Rewrite:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2 \right] + \gamma T$$

*Why rewrite by leaf:* each leaf's contribution is **independent**. We can optimize each leaf separately. And each leaf's contribution is a **quadratic** in $w_j$ — easy to minimize.

### Step 7: Solve for optimal leaf values

Each leaf's contribution is $G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2$. This is a parabola in $w_j$ (opens upward since $H_j + \lambda > 0$).

Take the derivative, set to zero:

$$\frac{\partial}{\partial w_j}\left[ G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2 \right] = G_j + (H_j + \lambda) w_j = 0$$

Solve:

$$\boxed{w_j^* = -\frac{G_j}{H_j + \lambda}}$$

Each piece:
- $-G_j$ (numerator): total correction direction. Minus sign = move opposite the gradient (downhill).
- $H_j$ (denominator): curvature. Larger $H$ = steeper landscape = take smaller step.
- $\lambda$ (denominator): regularization. Makes denominator bigger = shrinks leaf value toward zero.

### Step 8: Compute the optimal objective value

Plug $w_j^*$ back into the leaf's contribution:

$$G_j \cdot \left(-\frac{G_j}{H_j + \lambda}\right) + \frac{1}{2}(H_j + \lambda) \cdot \left(-\frac{G_j}{H_j + \lambda}\right)^2$$

$$= -\frac{G_j^2}{H_j + \lambda} + \frac{1}{2}\frac{G_j^2}{H_j + \lambda} = -\frac{1}{2}\frac{G_j^2}{H_j + \lambda}$$

Total optimal objective:

$$\boxed{\tilde{\mathcal{L}}^{(t)*} = -\frac{1}{2}\sum_{j=1}^{T}\frac{G_j^2}{H_j + \lambda} + \gamma T}$$

*What $G_j^2$ measures:* how aligned the gradients are in leaf $j$. If all data points need the same correction (large $|G_j|$) → $G_j^2$ is large → the leaf is very useful. If gradients cancel out ($G_j \approx 0$) → $G_j^2$ is small → the leaf is useless.

### Step 9: Split scoring (Gain)

To decide if splitting leaf $P$ into children $L$ and $R$ is worth it, compare the objective before and after:

$$\text{Before: } -\frac{1}{2}\frac{G_P^2}{H_P + \lambda} + \gamma \cdot T$$

$$\text{After: } -\frac{1}{2}\frac{G_L^2}{H_L + \lambda} - \frac{1}{2}\frac{G_R^2}{H_R + \lambda} + \gamma \cdot (T+1)$$

The gain is Before − After (we want to *decrease* the objective):

$$\boxed{\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G_P^2}{H_P + \lambda}\right] - \gamma}$$

- First three terms: improvement in objective from splitting (two specialized leaves vs. one general leaf)
- $\gamma$: cost of adding one more leaf. Gain $\leq 0$ → no split.

### Step 10: Summary — from objective to algorithm

| Derivation step | Result | Used in practice for |
|----------------|--------|---------------------|
| Taylor expansion | Replace loss with quadratic in $f_t$ | Avoid recomputing full loss — just need $g$, $h$ |
| Rewrite by leaf | Independent quadratic per leaf | Optimize each leaf separately |
| Solve quadratic | $w_j^* = -G_j / (H_j + \lambda)$ | Compute each leaf's correction |
| Plug $w_j^*$ back in | $-\frac{1}{2}G_j^2 / (H_j + \lambda)$ | Score how useful a leaf is |
| Compare before/after split | Gain formula | Decide whether to split |

Every hyperparameter maps to a specific line of math:
- $\lambda$ → denominator of $w_j^*$ and Gain
- $\gamma$ → subtracted from Gain
- $\eta$ → scales $f_t$ in the prediction update
- `max_depth` → limits how many times Step 9 recurses
- `min_child_weight` → minimum $H_j$ required to keep a leaf


## Section 4: The procedure — how one tree is added

At boosting step $b$, you already have predictions $\hat{y}_i^{(b-1)}$. You want to add one new tree that makes the loss smaller. Here's exactly what happens:

### Step 1: Compute how each point wants its prediction to change

Compute $g_i$ and $h_i$ for every data point (from the loss function).

### Step 2: Grow a tree that groups similar corrections

The tree partitions data points into leaves (regions) $R_1, R_2, ...$. Inside each leaf $j$, XGBoost applies the same correction value $w_j$ to every point in that leaf.

At each potential split, the tree asks: "is this split worth the complexity cost?" (see Step 4 below).

### Step 3: For each leaf, compute the best correction value

Aggregate the derivatives inside leaf $j$:

$$G_j = \sum_{i \in R_j} g_i \quad \text{(total "push" direction)}$$

$$H_j = \sum_{i \in R_j} h_i \quad \text{(total curvature)}$$

The optimal leaf value:

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

Three things are happening in this formula:
- **$-G_j$ (numerator):** "how much and which direction to correct" — comes from the Loss
- **$H_j$ (denominator):** "how carefully" — comes from the Loss curvature
- **$\lambda$ (denominator):** "be even more conservative" — comes from the Penalty term $\Omega$

**This is where $\lambda$ regularization actually happens.** It makes the denominator bigger, which shrinks every leaf value toward zero. You never "compute the penalty and add it to the loss" — $\lambda$ just lives in this denominator, pulling corrections back.

### Step 4: Decide splits using Gain (is the split worth it?)

When splitting a parent node $P$ into left $L$ and right $R$, you compare: one correction for everyone in $P$ vs. separate corrections for $L$ and $R$.

$$\text{Gain} = \frac{1}{2} \left( \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G_P^2}{H_P + \lambda} \right) - \gamma$$

Why $G^2$? Because you're measuring how good a leaf's best possible correction is. A leaf where gradients all point the same direction (large $|G|$) can make a strong, targeted correction — $G^2$ is large. A leaf where they cancel out ($G \approx 0$) is useless — $G^2$ is small.

**$\gamma$ is the second regularization parameter.** It's the cost of adding a leaf. Gain $\leq 0$ → no split. This prunes the tree *during construction*, not after.

So both regularization parameters dissolve into specific moments:
- **$\lambda$** shrinks every leaf value (Step 3)
- **$\gamma$** prevents weak splits (Step 4)

### Step 5: Update predictions with shrinkage

$$\hat{y}_i^{(b)} = \hat{y}_i^{(b-1)} + \eta \cdot f_b(x_i)$$

$\eta$ (learning rate) controls how much you trust each new tree. Smaller $\eta$ = safer updates, usually needs more trees. This is the third guardrail.

### Repeat Steps 1–5 for $B$ rounds.


## Section 5: Walk through with real numbers

Let's trace one complete round — from gradients to updated predictions.


In [1]:
import numpy as np
import pandas as pd

# Four F1 drivers — current predictions from previous trees
data = pd.DataFrame({
    'driver': ['Verstappen', 'Hamilton', 'Norris', 'Leclerc'],
    'actual_position': [1, 5, 3, 8],
    'current_prediction': [3.0, 4.0, 4.5, 5.5]
})

# STEP 1: Compute gradients
# For squared error: g = predicted - actual, h = 1
data['g_i'] = data['current_prediction'] - data['actual_position']
data['h_i'] = 1

print("STEP 1: Compute g and h for each data point")
print("=" * 60)
print(data.to_string(index=False))
print()
print("Verstappen: g = +2.0 → prediction is 2 too HIGH")
print("Leclerc:    g = -2.5 → prediction is 2.5 too LOW")
print("Hamilton:   g = -1.0 → prediction is 1 too LOW")
print("Norris:     g = +1.5 → prediction is 1.5 too HIGH")


STEP 1: Compute g and h for each data point
    driver  actual_position  current_prediction  g_i  h_i
Verstappen                1                 3.0  2.0    1
  Hamilton                5                 4.0 -1.0    1
    Norris                3                 4.5  1.5    1
   Leclerc                8                 5.5 -2.5    1

Verstappen: g = +2.0 → prediction is 2 too HIGH
Leclerc:    g = -2.5 → prediction is 2.5 too LOW
Hamilton:   g = -1.0 → prediction is 1 too LOW
Norris:     g = +1.5 → prediction is 1.5 too HIGH


In [2]:
# STEP 2-3: Tree groups drivers into leaves, compute leaf values
# The tree split on grid_position <= 3:
#   Leaf A: Verstappen + Norris (front-runners)
#   Leaf B: Hamilton + Leclerc (started further back)

lambda_reg = 1.0  # λ

# Leaf A
G_a = 2.0 + 1.5  # Verstappen(+2.0) + Norris(+1.5)
H_a = 1 + 1
w_a = -G_a / (H_a + lambda_reg)

# Leaf B
G_b = -1.0 + -2.5  # Hamilton(-1.0) + Leclerc(-2.5)
H_b = 1 + 1
w_b = -G_b / (H_b + lambda_reg)

print("STEP 3: Compute optimal leaf values")
print("=" * 60)
print(f"Leaf A (Verstappen, Norris):")
print(f"  G = {G_a}, H = {H_a}")
print(f"  w* = -{G_a} / ({H_a} + {lambda_reg}) = {w_a:.3f}")
print(f"  → Push predictions DOWN by {abs(w_a):.3f}")
print()
print(f"Leaf B (Hamilton, Leclerc):")
print(f"  G = {G_b}, H = {H_b}")
print(f"  w* = -({G_b}) / ({H_b} + {lambda_reg}) = {w_b:.3f}")
print(f"  → Push predictions UP by {w_b:.3f}")
print()

# Show regularization effect
w_a_no_reg = -G_a / H_a
print("--- Where λ does its work ---")
print(f"Without λ (λ=0): w_A = {w_a_no_reg:.3f}")
print(f"With λ=1:         w_A = {w_a:.3f}")
print(f"λ shrunk the correction by {abs(1 - w_a/w_a_no_reg)*100:.0f}%")
print("It's right there in the denominator — that IS regularization.")


STEP 3: Compute optimal leaf values
Leaf A (Verstappen, Norris):
  G = 3.5, H = 2
  w* = -3.5 / (2 + 1.0) = -1.167
  → Push predictions DOWN by 1.167

Leaf B (Hamilton, Leclerc):
  G = -3.5, H = 2
  w* = -(-3.5) / (2 + 1.0) = 1.167
  → Push predictions UP by 1.167

--- Where λ does its work ---
Without λ (λ=0): w_A = -1.750
With λ=1:         w_A = -1.167
λ shrunk the correction by 33%
It's right there in the denominator — that IS regularization.


In [ ]:
# STEP 4: Was the split worth it? Compute Gain
gamma = 1.0  # cost per leaf

G_p = G_a + G_b  # parent: all four drivers
H_p = H_a + H_b

score_left  = (G_a**2) / (H_a + lambda_reg)
score_right = (G_b**2) / (H_b + lambda_reg)
score_parent = (G_p**2) / (H_p + lambda_reg)

gain = 0.5 * (score_left + score_right - score_parent) - gamma

print("STEP 4: Is the split worth it?")
print("=" * 60)
print(f"Parent (all 4 drivers): G_P = {G_p}, score = {score_parent:.3f}")
print(f"  Gradients cancel out! Some need up, some need down.")
print(f"  This leaf can't make a useful correction.")
print()
print(f"Left  (Verstappen, Norris): G_L = {G_a}, score = {score_left:.3f}")
print(f"Right (Hamilton, Leclerc):  G_R = {G_b}, score = {score_right:.3f}")
print(f"  After split, each leaf has aligned gradients — targeted corrections!")
print()
print(f"Gain = 0.5 * ({score_left:.3f} + {score_right:.3f} - {score_parent:.3f}) - {gamma}")
print(f"     = {gain:.3f}")
print(f"  Gain > 0 → split is worth it! ✓")
print()
print(f"If γ were {score_left + score_right - score_parent + 1:.0f}, Gain would be negative → no split.")
print(f"That's how γ prunes the tree during construction.")


In [ ]:
# STEP 5: Update predictions with learning rate
eta = 0.1  # learning rate

data['tree_output'] = [w_a, w_b, w_a, w_b]
data['scaled_correction'] = data['tree_output'] * eta
data['new_prediction'] = data['current_prediction'] + data['scaled_correction']
data['new_error'] = data['actual_position'] - data['new_prediction']

print("STEP 5: Update predictions (η = 0.1)")
print("=" * 60)
cols = ['driver', 'actual_position', 'current_prediction', 
        'scaled_correction', 'new_prediction']
print(data[cols].to_string(index=False))
print()
print("Small steps! Verstappen: 3.0 → 2.88 (actual: 1)")
print("Next tree corrects a bit more. After 500 rounds, we converge.")
print()
print("THE FULL CHAIN:")
print("  g says 'Verstappen is predicted 2.0 too high'")
print("  → lands in Leaf A with Norris → G=3.5, H=2")
print("  → w* = -3.5/(2+1) = -1.17 (λ shrinks it)")
print("  → Gain check passes (γ doesn't block it)")
print("  → η scales to -0.117")
print("  → prediction drops 3.0 → 2.88")
print("  → repeat hundreds of times")


## Section 6: Connection to deep learning

If you've trained neural networks, the analogy is direct. Both follow "compute gradient → step downhill → regularize." The difference is *what gets updated*:

| | Deep Learning | XGBoost |
|---|---|---|
| **What you optimize** | Weights (numbers inside the network) | The ensemble (adding new trees) |
| **What gradient tells you** | "How should I adjust each weight?" | "How should each prediction change?" |
| **How you update** | Nudge weights: $w = w - \eta \cdot \nabla$ | Add a new tree that approximates the gradient |
| **Regularization** | $\lambda \sum w^2$ → shows up in weight update | $\gamma T + \frac{1}{2}\lambda \sum w_j^2$ → $\gamma$ gates splits, $\lambda$ shrinks leaves |
| **Previous state** | Weights are modified in place | Previous trees are frozen, can only add new ones |

The penalty works the same way conceptually: in deep learning, L2 regularization shows up in the weight update rule. In XGBoost, the penalty dissolves into the leaf value formula ($\lambda$) and split scoring ($\gamma$). Same idea, different mechanism.


## Section 7: Regression vs. Classification

The framework stays the same — only the loss function changes, which changes $g$ and $h$.

**Regression** (squared error):
- $g_i = \hat{y}_i - y_i$, $h_i = 1$
- Final prediction = sum of tree outputs

**Classification** (logistic loss):
- Model outputs a score (logit); probability is $p(y=1|x) = \sigma(\hat{f}(x))$
- $g_i = p_i - y_i$, $h_i = p_i(1 - p_i)$
- Note: $h_i$ is no longer constant — it varies by data point, which means the hessian actually matters here (unlike squared error where it's always 1)

Everything else — the leaf formula, the gain calculation, the learning rate — stays identical. That's the power of the gradient framework: swap the loss, and the entire algorithm adapts.


## Section 8: Experiments

Now let's use the real XGBoost library and verify everything we just learned.


In [ ]:
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# F1 dataset
n = 600
grid_position = np.random.randint(1, 21, n)
team_strength = np.random.uniform(0.3, 1.0, n)
is_wet = np.random.binomial(1, 0.2, n)
tire_compound = np.random.choice([0, 1, 2], n)
driver_skill = np.random.uniform(0.5, 1.0, n)

finish_position = (
    grid_position * 0.5 + (1 - team_strength) * 6 
    + (1 - driver_skill) * 4 - is_wet * 1.5
    + tire_compound * 0.5 + np.random.normal(0, 2, n)
).clip(1, 20)

df = pd.DataFrame({
    'grid_position': grid_position, 'team_strength': team_strength,
    'is_wet': is_wet, 'tire_compound': tire_compound,
    'driver_skill': driver_skill, 'finish_position': finish_position
})

X = df.drop('finish_position', axis=1)
y = df['finish_position']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")


### Experiment 1: Single tree vs. random forest vs. XGBoost


In [ ]:
single_tree = DecisionTreeRegressor(max_depth=5, random_state=42)
single_tree.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)

print("Model Comparison")
print("=" * 60)
for name, model in [('Single Tree', single_tree), ('Random Forest', rf), ('XGBoost', xgb_model)]:
    tr = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    te = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    print(f"{name:15s}  Train RMSE: {tr:.3f}  Test RMSE: {te:.3f}  Gap: {tr-te:.3f}")


### Experiment 2: Intentionally overfit, then fix it

The order of fixes matters — this is what interviewers want to hear.


In [ ]:
overfit = xgb.XGBRegressor(
    n_estimators=500, max_depth=10, learning_rate=0.3,
    subsample=1.0, colsample_bytree=1.0, reg_lambda=0, reg_alpha=0,
    random_state=42, verbosity=0
)
overfit.fit(X_train, y_train)
tr = np.sqrt(mean_squared_error(y_train, overfit.predict(X_train)))
te = np.sqrt(mean_squared_error(y_test, overfit.predict(X_test)))
print(f"Overfit model: Train {tr:.3f} | Test {te:.3f} | Gap {tr-te:.3f}")

fixes = {
    'Fix 1: depth 10→4':        dict(max_depth=4, learning_rate=0.3, subsample=1.0, colsample_bytree=1.0, reg_lambda=0, reg_alpha=0),
    'Fix 2: + LR 0.3→0.1':      dict(max_depth=4, learning_rate=0.1, subsample=1.0, colsample_bytree=1.0, reg_lambda=0, reg_alpha=0),
    'Fix 3: + subsample 0.8':    dict(max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_lambda=0, reg_alpha=0),
    'Fix 4: + regularization':   dict(max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.1),
}

for name, params in fixes.items():
    m = xgb.XGBRegressor(n_estimators=500, random_state=42, verbosity=0, **params)
    m.fit(X_train, y_train)
    tr = np.sqrt(mean_squared_error(y_train, m.predict(X_train)))
    te = np.sqrt(mean_squared_error(y_test, m.predict(X_test)))
    print(f"{name}: Train {tr:.3f} | Test {te:.3f} | Gap {tr-te:.3f}")

print("\nInterview order: depth → learning rate → subsampling → regularization")


### Experiment 3: Early stopping


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

model_es = xgb.XGBRegressor(
    n_estimators=1000, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, verbosity=0, early_stopping_rounds=20
)
model_es.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

print(f"Set 1000 trees, early stopping used {model_es.best_iteration}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, model_es.predict(X_test))):.3f}")
print()
print("'How do you choose the number of trees?'")
print("→ Always early stopping. Never manual tuning.")


### Experiment 4: Feature importance


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, imp_type in zip(axes, ['weight', 'gain', 'cover']):
    scores = xgb_model.get_booster().get_score(importance_type=imp_type)
    pd.Series(scores).sort_values().plot(kind='barh', ax=ax)
    ax.set_title(f'Importance by {imp_type}')
plt.tight_layout()
plt.show()

print("weight = # times used in splits")
print("gain   = avg loss improvement per split ← MOST MEANINGFUL")
print("cover  = avg # samples affected per split")
print("For rigorous importance → use SHAP values (covered later)")


### Experiment 5: Missing value handling


In [ ]:
X_train_missing = X_train.copy()
X_test_missing = X_test.copy()
mask = np.random.random(len(X_train_missing)) < 0.15
X_train_missing.loc[mask, 'team_strength'] = np.nan
mask_test = np.random.random(len(X_test_missing)) < 0.15
X_test_missing.loc[mask_test, 'team_strength'] = np.nan

xgb_missing = xgb.XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=42, verbosity=0
)
xgb_missing.fit(X_train_missing, y_train)

clean = np.sqrt(mean_squared_error(y_test, xgb_model.predict(X_test)))
missing = np.sqrt(mean_squared_error(y_test, xgb_missing.predict(X_test_missing)))

print(f"Clean data RMSE:   {clean:.3f}")
print(f"15% missing RMSE:  {missing:.3f}")
print()
print("XGBoost handles NaNs natively — at each split, it tries sending")
print("missing values left and right, picks whichever reduces loss more.")


## Section 9: Hyperparameters — mapped to what you now know

Every hyperparameter connects to something in the procedure above:

| Parameter | What it does | Where in the procedure | Typical range |
|-----------|-------------|----------------------|---------------|
| `learning_rate` ($\eta$) | Scales each tree's contribution | Step 5: $\hat{y} += \eta \cdot f_b$ | 0.01–0.1 |
| `max_depth` | How complex each tree can be | Step 2: limits tree growth | 3–6 |
| `n_estimators` ($B$) | Number of boosting rounds | How many times Steps 1–5 repeat | Use early stopping |
| `subsample` | Fraction of rows per tree | Step 2: adds randomness | 0.7–0.9 |
| `colsample_bytree` | Fraction of features per tree | Step 2: adds randomness | 0.7–0.9 |
| `reg_lambda` ($\lambda$) | Shrinks leaf values toward zero | Step 3: denominator of $w_j^*$ | 1–10 |
| `reg_alpha` ($\alpha$) | Pushes small leaf values to exactly zero | Step 3: L1 penalty on $w_j$ | 0–1 |
| `gamma` ($\gamma$) | Minimum improvement to justify a split | Step 4: subtracted from Gain | 0–5 |
| `min_child_weight` | Min sum of $h_i$ in a leaf | Step 2: prevents tiny/noisy splits | 1–10 |

**Best practice:** Use `early_stopping_rounds` to stop when validation metric stops improving.

**Metrics:**
- Regression: RMSE, MAE
- Classification: logloss, ROC-AUC, PR-AUC (use PR-AUC if classes are imbalanced)
